In [47]:
import mlflow

In [48]:
mlflow.set_experiment("For promote")

2026/10/08 20:33:15 INFO mlflow.tracking.fluent: Experiment with name 'For promote' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/himmat/revision_repo/mlruns/6', creation_time=1791470895074, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1791470895074, lifecycle_stage='active', name='For promote', tags={}, trace_location=None, workspace='default'>

In [49]:
from sklearn.datasets import load_iris

In [50]:
data = load_iris()

In [51]:
from sklearn.model_selection import train_test_split

x_train, x_temp, y_train, y_temp = train_test_split(data.data, data.target, test_size=0.3, random_state=42)

In [52]:
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [54]:
from mlflow.client import MlflowClient

In [55]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score
)

In [56]:
with mlflow.start_run(run_name="LR-Model") as lrrun:
    lr = LogisticRegression()
    lr.fit(x_train, y_train)
    y_pred  = lr.predict(x_val)
    metrics = {
        "accuracy":  accuracy_score(y_val, y_pred),
        "precision": precision_score(
            y_val, y_pred, average="weighted"
        ),
        "recall": recall_score(
            y_val, y_pred, average="weighted"
        ),
        "f1": f1_score(
            y_val, y_pred, average="weighted"
        )
    }
    mlflow.log_metrics(metrics)
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="model",
        registered_model_name="lr-model"
    )

2026/10/08 20:33:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/10/08 20:33:15 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/himmat/revision_repo
2026/10/08 20:33:15 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/himmat/revision_repo
2026/10/08 20:33:15 INFO mlflow.utils.environment: Detected uv project at /Users/himmat/revision_repo. Attempting to export requirements via 'uv export'.
2026/10/08 20:33:15 INFO mlflow.utils.uv_utils: Exported 199 dependencies via uv
2026/10/08 20:33:15 INFO mlflow.utils.environment: Successfully exported 199 requirements from uv project. Skipping package capture based inference.
2026/10/08 20:33:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'lr-model' already exists. Creating a new version of this model...
Created version '3' of model 'lr-model'.


In [57]:
client = MlflowClient()

In [58]:
run_id = lrrun.info.run_id

In [59]:
version = client.search_model_versions(f"run_id='{run_id}'")[0].version
client.set_registered_model_alias("lr-model", "champion", version)

In [60]:
with mlflow.start_run(run_name="KNN-model") as run:
    knn = LogisticRegression()
    knn.fit(x_train, y_train)
    y_pred  = knn.predict(x_val)
    metrics = {
        "accuracy":  accuracy_score(y_val, y_pred),
        "precision": precision_score(
            y_val, y_pred, average="weighted"
        ),
        "recall": recall_score(
            y_val, y_pred, average="weighted"
        ),
        "f1": f1_score(
            y_val, y_pred, average="weighted"
        )
    }
    mlflow.log_metrics(metrics)
    model_info = mlflow.sklearn.log_model(
        sk_model=knn,
        artifact_path="model",
        registered_model_name="knn-model"
    )

2026/10/08 20:33:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/10/08 20:33:15 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/himmat/revision_repo
2026/10/08 20:33:15 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/himmat/revision_repo
2026/10/08 20:33:15 INFO mlflow.utils.environment: Detected uv project at /Users/himmat/revision_repo. Attempting to export requirements via 'uv export'.
2026/10/08 20:33:15 INFO mlflow.utils.uv_utils: Exported 199 dependencies via uv
2026/10/08 20:33:15 INFO mlflow.utils.environment: Successfully exported 199 requirements from uv project. Skipping package capture based inference.
2026/10/08 20:33:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'knn-model' already exists. Creating a new version of this model...
Created version '4' of model 'knn-model'.


In [61]:
run = run.info.run_id

In [63]:
version = client.search_model_versions(f"run_id='{run_id}'")[0].version
client.set_registered_model_alias("lr-model", "challenger", version)

In [74]:
def evaluate(name, alias):
    model = mlflow.pyfunc.load_model(f"models:/{name}@{alias}")
    p = model.predict(x_test)
    return {"precision": precision_score(y_test, p, average="weighted"),
            "recall": recall_score(y_test, p, average="weighted"),
            "f1": f1_score(y_test, p, average="weighted")}

In [76]:
chall = evaluate(name="knn-model", alias="challenger")
print("challenger", chall)

challenger {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [77]:
champ = evaluate(name = "lr-model", alias="champion")
print("champion", champ)

champion {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [78]:
from mlflow.exceptions import MlflowException

In [ ]:
try:
    champ = evaluate(name = "lr-model", alias="champion") 
    print("champion:  ", champ)
    better = (
        chall["f1"] > champ["f1"] + 0.002 and
        chall["precision"] >= champ["precision"] - 0.005 and   # don't flag real emails
        chall["recall"] >= champ["recall"] - 0.01
    
except MlflowException:                    # no champion yet → first model
    champ, better = None, chall["precision"] >= 0.97

champion:   {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [84]:
better

False

In [ ]:
v = client.get_model_version_by_alias("knn-model", "challenger").version

if better:
    if champ:  # keep rollback pointer
        old = client.get_model_version_by_alias('lr-model', "champion").version
        client.set_registered_model_alias('lr-model', "previous", old)
    client.set_registered_model_alias('knn-model', "champion", v)     # PROMOTE
    client.set_model_version_tag('knn-model', v, "validation", "promoted")
    print(f"v{v} promoted to champion")
else:
    client.set_model_version_tag('lr-model', v, "validation", "rejected")
    print(f"v{v} rejected, champion unchanged")

client.delete_registered_model_alias('knn-model', "challenger")